# L12c: Deep Q-Learning
In this lecture, we replace the Q-table with a function that computes the action values from the state. This idea is the basis of deep Q-learning and of its best-known method, the deep Q-network (DQN). However, today we will start with the simplest such function, a linear one.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Explain the limits of a Q-table:__ A Q-table needs one entry for every state and action, and the number of states grows exponentially with the number of state variables. A table also learns nothing about a state from visits to nearby states.
> * __Train a Q-network with gradient descent:__ Deep Q-learning moves each Q-value toward the reward plus the discounted value of the next state. We'll derive the gradient step that updates the weights of a linear network.
> * __Explain the replay buffer and the target network:__ These two parts keep the training of a Q-network from becoming unstable. The replay buffer trains on random samples of past experience, and the target network keeps the next-state values fixed for a while.

Let's get started!
___

## Examples
We will use the following example to illustrate the key ideas of this lecture:

> [▶ Learn a dosing policy without a grid using a linear DQN](CHEME-5800-L12c-Example-LinearDQN-AdaptiveDosing-Fall-2026.ipynb). In this example, an agent learns to dose the patient of the dosing lab without rounding the tumor burden and the toxicity to grid levels. The agent's linear Q-network has far fewer weights than the dosing lab's Q-table has entries. We compare its doses with value iteration on a much finer grid.

___

## The Curse of Dimensionality
__The Q-table.__ Entry $(i,j)$ holds the value (utility) of taking action $j$ in state $i$. The policy reads each row and picks the column with the largest entry. For state $s$, that is action $k$.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
<img class="course-diagram" src="../L12a/figs/Fig-Q-Schematic.svg" width="580" style="max-width:100%; height:auto;" alt="A grid of dots with states as rows and actions as columns. The dot in row state i and column action j is highlighted as utility (i,j). In the row for state s, the dot in column action k is highlighted as the maximum utility for state s."/>
</p>

The Q-table has three problems:

* __The table grows exponentially.__ A state with $m$ variables, each taking $L$ values, has $L^{m}$ states, a growth that Richard Bellman called the [curse of dimensionality](https://en.wikipedia.org/wiki/Curse_of_dimensionality). The grid world has two coordinates with 20 values each, or $20^{2}=400$ states. Adding a battery level with 20 values would give $20^{3}=8000$ states.
* __The table does not generalize.__ Each entry learns only from visits to its own state and action. What the agent learns in one state tells it nothing about the states next to it. In the grid world, the roomba must visit every square to learn its value.
* __Rounding changes the problem.__ A table needs a finite list of states. Rounding a continuous variable to grid levels treats different states as the same state. In the dosing lab, the policy gives no dose at a tumor burden of 0.109, because the grid rounds it to 0.100.

Instead, we replace the table with a function $Q_{\theta}:\mathcal{S}\times\mathcal{A}\rightarrow\mathbb{R}$ with parameters $\theta$. Its value $Q_{\theta}(s,a)$ is an estimate of the action value $Q(s,a)$.

Each state is a vector of $m$ state variables, $s\in\mathcal{S}\subseteq\mathbb{R}^{m}$, such as the roomba's two coordinates. We do not need to enumerate the states.

___

## Q-Networks
A __Q-network__ computes $Q_{\theta}(s,a)$ for every action at once. It takes a state $s\in\mathcal{S}$ as its input and returns the vector $\left(Q_{\theta}(s,a_{1}),\dots,Q_{\theta}(s,a_{n})\right)\in\mathbb{R}^{n}$, one value for each of the $n$ actions in the action set $\mathcal{A}=\{a_{1},\dots,a_{n}\}$. The greedy policy picks the action with the largest value, $\pi(s)=\arg\max_{a\in\mathcal{A}}Q_{\theta}(s,a)$.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-Q-Networks/Fig-Q-Networks.svg" width="720" style="max-width:100%; height:auto;" alt="Two diagrams, each mapping a state s to one Q-value per action, Q of s and a 1 through Q of s and a n. The top diagram, labeled linear Q-network, sends the state through gray arrows to a column of feature circles phi 1, phi 2, down to phi p, labeled features phi of s, fixed. Blue lines, labeled weights W, learned, connect every feature to every Q-value. The bottom diagram, labeled deep Q-network, sends the state into a shaded box of three hidden layers of circles joined by blue lines, labeled hidden layers, weights theta, learned. One arrow leaves the box and fans out to the Q-values."/>
</p>

__Two Q-networks.__ In both networks, training estimates only the weights. We choose everything else:

* __Linear network:__ We choose the $p$ features $\phi_{1}(s),\dots,\phi_{p}(s)$, fixed functions of the state. Training estimates the weights from the features to the Q-values (blue).
* __Deep network:__ We choose the number of hidden layers, the number of units in each layer, and their functions. Training estimates the weights of every layer.

Training now adjusts the weights $\theta$ instead of the entries of a table. The number of weights depends on the features, layers, and actions we choose, not on the number of states.

### A linear Q-network
A linear Q-network first maps the state $s$ to a __feature vector__ $\phi(s)=\left(\phi_{1}(s),\dots,\phi_{p}(s)\right)\in\mathbb{R}^{p}$. Each feature $\phi_{j}:\mathcal{S}\rightarrow\mathbb{R}$ is a fixed function of the state that we choose. The Q-value of action $a$, a weighted sum of the features with weight vector $\mathbf{w}_{a}\in\mathbb{R}^{p}$, is given by:

$$
\begin{equation*}
Q_{\mathbf{W}}(s,a) = \mathbf{w}_{a}^{\top}\phi(s) = \sum_{j=1}^{p}w_{a,j}\,\phi_{j}(s),
\end{equation*}
$$

where $w_{a,j}$ is the weight of action $a$ on feature $\phi_{j}(s)$. The weight matrix $\mathbf{W}$ has one row $\mathbf{w}_{a}^{\top}$ for each of the $n$ actions, which gives $n\cdot p$ weights in all.

__The features.__ In the example, the state is the pair $s=(B,Z)$ of tumor burden and toxicity. Its features are __radial basis functions__, with centers $\mathbf{c}_{j}$ on an even grid over the square $0\leq B\leq 1$, $0\leq Z\leq 1$. Feature $j$ is given by:

$$
\begin{equation*}
\phi_{j}(s)=\exp\left(-\frac{\lVert s-\mathbf{c}_{j}\rVert_{2}^{2}}{2\sigma^{2}}\right),
\end{equation*}
$$

where $\lVert s-\mathbf{c}_{j}\rVert_{2}$ is the Euclidean distance from the state to center $j$, and $\sigma$ sets the width of every basis function. Basis function $j$ equals one at its center and falls toward zero away from it. Nearby states have similar features. A weight update at one state also changes the Q-values of the states near it.

We choose the centers and the width $\sigma$ before training, and they stay fixed. A simple choice puts the centers on an even grid over the range of each state variable. With $\sigma$ close to the grid spacing, neighboring basis functions overlap.

An even grid with $L$ centers per variable needs $L^{m}$ centers for $m$ state variables, which brings back the curse of dimensionality.

__A Q-table is a linear network.__ A linear network with one feature per state reproduces the Q-table. Suppose there are $K$ states, numbered $s_{1},\dots,s_{K}$, and give the network $p=K$ features:

* __One-hot features:__ The feature vector of state $s_{k}$ is the __one-hot__ vector $\phi(s_{k})=\mathbf{e}_{k}$, with a one in entry $k$ and zeros everywhere else.
* __The dot product picks one weight:__ The Q-value is $Q_{\mathbf{W}}(s_{k},a)=\mathbf{w}_{a}^{\top}\mathbf{e}_{k}=w_{a,k}$, the weight of action $a$ on feature $k$.
* __Each weight is one table entry:__ The weight $w_{a,k}$ holds the value of action $a$ in state $s_{k}$, the same number as the Q-table entry $Q(s_{k},a)$. The Q-table $\mathbf{Q}$ stores this number in row $k$ and column $a$, while $\mathbf{W}$ stores it in row $a$ and column $k$. Thus, $\mathbf{W}=\mathbf{Q}^{\top}$.

### A deep Q-network
A __feedforward neural network__ has an input layer, one or more __hidden layers__, and an output layer. Information flows from the input to the output, with no cycles. A network with more than one hidden layer is called __deep__.

* __Input layer:__ It holds the state variables $s\in\mathbb{R}^{m}$ and does no computation.
* __Hidden layers:__ Each node, a circle in the figure, computes a weighted sum of the outputs of the layer before it, plus a bias. It then applies a nonlinear __activation function__, such as $\text{ReLU}(x)=\max(0,x)$.
* __Output layer:__ Each of the $n$ output nodes computes a weighted sum of the last hidden layer's outputs. It applies no activation function. Thus, a Q-value can take any sign.

The network is a composition of these layer functions. The parameters $\theta$ are the weights and biases of every layer.

[Mnih et al. (2015)](https://www.nature.com/articles/nature14236) trained a deep Q-network to play Atari video games from the pixels on the screen. The rest of this lecture holds for any Q-network with parameters $\theta$, the weights $\mathbf{W}$ in the linear case.

___

## The DQN Loss
The training data are __transitions__ $(s_{i},a_{i},r_{i},s_{i}^{\prime},d_{i})$: a state, the action taken, the reward, the next state, and a __done flag__ $d_{i}$. The flag is one if $s_{i}^{\prime}$ is __absorbing__, a state that ends the episode, such as a cure or a toxic death, and zero otherwise. An episode that stops at a step limit keeps $d_{i}=0$ on its last transition. 

> __DQN Bellman Loss:__
>
> The target $y_{i}$ of transition $i$ is the value that the prediction $Q_{\theta}(s_{i},a_{i})$ should match. It is the reward plus the discounted value of the next state:
>
> $$y_{i} = r_{i} + \gamma\,(1-d_{i})\max_{a^{\prime}\in\mathcal{A}}Q_{\theta^{-}}(s_{i}^{\prime},a^{\prime}),$$
>
> where $\theta^{-}$ are the parameters of the __target network__, a delayed copy of the Q-network (we explain in the next section), and $\gamma$ is the discount factor. The factor $(1-d_{i})$ drops the future value when $s_{i}^{\prime}$ is absorbing, which leaves $y_{i}=r_{i}$.
>
> The difference $y_{i}-Q_{\theta}(s_{i},a_{i})$ is the __prediction error__ of transition $i$. The network trains on a __mini-batch__, a small random sample of $N_{\text{batch}}$ past transitions, where $\mathcal{L}(\theta)$ is the loss function:
>
> $$\mathcal{L}(\theta) = \frac{1}{2N_{\text{batch}}}\sum_{i=1}^{N_{\text{batch}}}\Bigl(\underbrace{y_{i}-Q_{\theta}(s_{i},a_{i})}_{\text{prediction error}}\Bigr)^{2}.$$
>
> The prediction uses the action $a_{i}$ that was actually taken. The target uses the best next action under the target network.

Training uses __gradient descent__. One step with learning rate $\alpha$, which sets the size of the step, is $\theta\gets\theta-\alpha\nabla_{\theta}\mathcal{L}(\theta)$, where the gradient $\nabla_{\theta}\mathcal{L}$ is the vector of derivatives of the loss with respect to the parameters.

### The gradient step of a linear network
For the linear network, we need the gradient of the loss with respect to the weight vector $\mathbf{w}_{a}$ of each action $a$. Two facts make this gradient simple:

* __The prediction is linear in the weights:__ The prediction of transition $i$ is $Q_{\mathbf{W}}(s_{i},a_{i})=\mathbf{w}_{a_{i}}^{\top}\phi(s_{i})$. Its gradient with respect to $\mathbf{w}_{a}$ is $\phi(s_{i})$ if $a_{i}=a$, and zero otherwise.
* __The target does not depend on $\mathbf{W}$:__ The target $y_{i}=r_{i}+\gamma\,(1-d_{i})\max_{a^{\prime}}Q_{\mathbf{W}^{-}}(s_{i}^{\prime},a^{\prime})$ uses the weights $\mathbf{W}^{-}$ of the target network, a separate copy of $\mathbf{W}$. Thus, the gradient of $y_{i}$ with respect to $\mathbf{w}_{a}$ is zero.

Substituting the linear prediction into the loss gives $\mathcal{L}(\mathbf{W})=\frac{1}{2N_{\text{batch}}}\sum_{i=1}^{N_{\text{batch}}}\left(y_{i}-\mathbf{w}_{a_{i}}^{\top}\phi(s_{i})\right)^{2}$. Its gradient with respect to $\mathbf{w}_{a}$ follows in three steps:

$$
\begin{align*}
\nabla_{\mathbf{w}_{a}}\mathcal{L} &= \frac{1}{2N_{\text{batch}}}\sum_{i=1}^{N_{\text{batch}}}\nabla_{\mathbf{w}_{a}}\left(y_{i}-\mathbf{w}_{a_{i}}^{\top}\phi(s_{i})\right)^{2} && \text{gradient of each term in the sum}\\
&= \frac{1}{2N_{\text{batch}}}\sum_{i=1}^{N_{\text{batch}}}2\left(y_{i}-\mathbf{w}_{a_{i}}^{\top}\phi(s_{i})\right)\left(-\nabla_{\mathbf{w}_{a}}\,\mathbf{w}_{a_{i}}^{\top}\phi(s_{i})\right) && \text{chain rule, } y_{i} \text{ does not depend on } \mathbf{W}\\
&= -\frac{1}{N_{\text{batch}}}\sum_{i\,:\,a_{i}=a}\left(y_{i}-\mathbf{w}_{a}^{\top}\phi(s_{i})\right)\phi(s_{i}) && \nabla_{\mathbf{w}_{a}}\,\mathbf{w}_{a_{i}}^{\top}\phi(s_{i})=\phi(s_{i}) \text{ if } a_{i}=a \text{, else } 0
\end{align*}
$$

where the last sum runs only over the transitions $i$ in the mini-batch whose action $a_{i}$ equals $a$. For example, if a mini-batch of four transitions has the actions $(a_{2},a_{1},a_{2},a_{3})$, the gradient for $\mathbf{w}_{a_{2}}$ adds up transitions $i=1$ and $i=3$ only. The factor $\tfrac{1}{2}$ in the loss cancels the 2 from the derivative of the square.

One gradient step, $\mathbf{w}_{a}\gets\mathbf{w}_{a}-\alpha\nabla_{\mathbf{w}_{a}}\mathcal{L}$, updates the row of each action in the mini-batch. The update of row $a$ is given by:

$$
\begin{equation*}
\mathbf{w}_{a}\gets\mathbf{w}_{a}+\frac{\alpha}{N_{\text{batch}}}\sum_{i\,:\,a_{i}=a}\left(y_{i}-\mathbf{w}_{a}^{\top}\phi(s_{i})\right)\phi(s_{i}).
\end{equation*}
$$

In one training step, we first compute the prediction error of every transition in the mini-batch with the current weights. We then update each row $\mathbf{w}_{a}$ once, using the errors of the transitions that took action $a$.

___

## DQN Algorithm and Stabilizers
Tabular Q-learning converges if it visits every state-action pair often enough. Deep Q-learning has no such guarantee, and its Q-values can even grow without bound, a failure called __divergence__ ([Sutton and Barto, 2018, Chapter 11](http://incompleteideas.net/book/the-book-2nd.html)). Two additions make divergence much less likely:

__The replay buffer:__  The __replay buffer__ $\mathcal{D}$ is the agent's memory of its experience. It stores the most recent $M$ transitions, and these transitions are the training data.
* __How we use it:__ Each training step draws a random mini-batch of $N_{\text{batch}}$ transitions from the buffer. A random draw mixes old and new experience from many episodes, instead of only the last few steps. Each transition is also used in many training steps.
* __The warm-up:__ Training waits until the buffer holds $N_{\text{warm}}$ transitions, a period called the __warm-up__. Without it, the first mini-batches would hold only a few similar transitions.

__The target network:__ The __target network__ $Q_{\theta^{-}}(s,a)$ is a frozen copy of the Q-network. It computes the targets $y_{i}$, and training does not change its weights $\theta^{-}$.
* __How we use it:__ Every $C$ steps, counted over all episodes, the algorithm copies the Q-network into the target network, $\theta^{-}\gets\theta$. Between copies, the targets stay fixed while the Q-network trains.
* __Why we need it:__ Without it, the Q-network would compute its own targets. Each gradient step would then move the targets as well as the predictions.

### Algorithm
The DQN algorithm runs in episodes, like Q-learning.

__Initialize__: Given a Q-network $Q_{\theta}(s,a)$, set $\theta$ to zeros for a linear network or to small random values for a deep network, and copy the parameters to the target network, $\theta^{-}\gets\theta$. Start with an empty replay buffer $\mathcal{D}$ and a total step counter $\tau\gets 0$. 

Hyperparameters: the discount factor $0\leq\gamma<1$, the learning rate $\alpha>0$, the exploration probability $\epsilon$ of each episode, the number of episodes $N$, the maximum number of steps per episode $t_{\max}$, the buffer size $M$, the warm-up size $N_{\text{warm}}$ (the number of transitions the buffer must hold before training begins), the mini-batch size $N_{\text{batch}}$, and the target-network copy interval $C$.

For each of the $N$ episodes __do__:
1. Choose a starting state $s$ that is not absorbing, and set the episode step counter $t\gets 0$.
2. While $s$ is not absorbing and $t<t_{\max}$ __do__:
    1. Choose the action $a$: draw a random number $\rho$ uniformly from $[0,1]$.
        - If $\rho\leq\epsilon$ __then__ choose $a\in\mathcal{A}$ uniformly at random (exploration).
        - Otherwise, choose the greedy action $a\gets\arg\max_{a^{\prime}\in\mathcal{A}}Q_{\theta}(s,a^{\prime})$ (exploitation).
    2. Take action $a$, and observe the reward $r$, the next state $s^{\prime}$, and the done flag $d$ from the _world_, the simulation or system the agent acts on, where $d=1$ if $s^{\prime}$ is absorbing and $d=0$ otherwise.
    3. Store the transition $(s,a,r,s^{\prime},d)$ in the replay buffer $\mathcal{D}$, and remove the oldest transition if $\mathcal{D}$ holds more than $M$.
    4. Update the counters, $t\gets t+1$ and $\tau\gets\tau+1$.
    5. If $\mathcal{D}$ holds at least $N_{\text{warm}}$ transitions __then__ take one training step:
        1. Draw a mini-batch of $N_{\text{batch}}$ transitions $(s_{i},a_{i},r_{i},s_{i}^{\prime},d_{i})$, $i=1,\dots,N_{\text{batch}}$, uniformly at random with replacement from $\mathcal{D}$.
        2. Compute the target of each transition with the _target network_, $y_{i}\gets r_{i}+\gamma\,(1-d_{i})\max_{a^{\prime}\in\mathcal{A}}Q_{\theta^{-}}(s_{i}^{\prime},a^{\prime})$.
        3. Compute the loss on the actions that were actually taken:
           $$\mathcal{L}(\theta)=\frac{1}{2N_{\text{batch}}}\sum_{i=1}^{N_{\text{batch}}}\left(y_{i}-Q_{\theta}(s_{i},a_{i})\right)^{2}.$$
        4. Take a _single_ gradient descent step on the loss, $\theta\gets\theta-\alpha\nabla_{\theta}\mathcal{L}(\theta)$.
            - _Why only a single step?_ Many steps on one small mini-batch can overfit its few transitions and undo what the network learned from the rest of the buffer.
    6. If $\tau$ is a multiple of $C$ __then__ copy the Q-network to the target network, $\theta^{-}\gets\theta$.
    7. Update the state, $s\gets s^{\prime}$.

__Extract Policy__: For any state $s$, compute $\pi(s)\gets\arg\max_{a\in\mathcal{A}}Q_{\theta}(s,a)$.

The policy gives an action for any state, including states the agent never visited.

__Why random starting weights for a deep network?__ If all the weights start equal, the units of a hidden layer compute the same feature, get the same update, and stay equal. A linear network has fixed features, and zeros work.

In the example, a linear network with radial basis features trains quickly on a laptop. The same algorithm trains a deep network, which needs a package for neural networks and much more time.

> __Example:__
> 
> [▶ Learn a dosing policy without a grid using a linear DQN](CHEME-5800-L12c-Example-LinearDQN-AdaptiveDosing-Fall-2026.ipynb). In this example, an agent learns to dose the patient of the dosing lab without rounding the tumor burden and the toxicity to grid levels. The agent's linear Q-network has far fewer weights than the dosing lab's Q-table has entries. We compare its doses with value iteration on a much finer grid.

### Overestimation
The DQN target uses the largest estimated Q-value of the next state. Every estimate has some error, and the largest one is often an estimate whose error is positive. The target then tends to be too high on average, even when each estimate is right on average.

This upward bias is called __overestimation__. Each target uses the value of the next state, which passes the bias back to earlier states. The bias can make a worse action look like the best one.

The maximum uses the same estimates to choose the next action and to score it. __Double DQN__ ([van Hasselt, Guez, and Silver, 2016](https://arxiv.org/abs/1509.06461)) chooses with the Q-network and scores with the target network. Its target is given by:

$$
\begin{equation*}
y_{i} = r_{i} + \gamma\,(1-d_{i})\,Q_{\theta^{-}}\left(s_{i}^{\prime},\;\arg\max_{a^{\prime}\in\mathcal{A}}Q_{\theta}(s_{i}^{\prime},a^{\prime})\right).
\end{equation*}
$$

The target network is a recent copy of the Q-network, and the two share part of their errors. Double DQN can reduce overestimation, but it does not always remove it.

___

## Lab
In the [lab](../L12d/CHEME-5800-L12d-Lab-DoubleDQN-Fall-2026.ipynb), we'll measure overestimation and watch it make Q-learning and DQN prefer the worse action on a small problem with two states. We'll then test two fixes that choose with one estimate and score with another: Double Q-learning, which keeps two Q-tables that score each other's choices, and Double DQN.

___

## Summary
In this lecture, we replaced the Q-table with a Q-network and built deep Q-learning around it.

> __Key Takeaways:__
>
> * __A Q-network replaces the table:__ A table needs one entry per state and action and learns nothing about a state from its neighbors. A Q-network computes every action value from features of the state, with a number of parameters that does not grow with the number of states.
>
> * __Deep Q-learning trains the network with gradient descent:__ Each target is the reward plus the discounted best value of the next state under the target network, or the reward alone when the next state is absorbing. For a linear network, one gradient step adds each prediction error times the features to the weights of the action taken.
>
> * __The replay buffer and the target network stabilize training:__ Random mini-batches from the replay buffer mix transitions from many episodes and reuse each one many times. The target network changes only when the Q-network is copied into it, which keeps the targets fixed between copies.

A linear network with well-chosen features is often enough for a problem with a few state variables. In a deep network, the features come from hidden-layer weights estimated from data. This lets the same algorithm work from raw inputs such as images.

___